# Chapter 08 - Type Hints

#### 1. Mechanical Refresher
Type hints attach expected types to names, parameters, and return values, but Python does not enforce them at runtime by default. They make signatures easier to read, help editors and static checkers catch mismatches, and document production code contracts such as `list[float]`, `str | None`, `Callable`, and typed dictionaries or containers.

#### 2. Minimal Working Example

In [ ]:
from typing import Callable

def apply_metric(metric: Callable[[list[float], list[float]], float],
                 predictions: list[float],
                 targets: list[float]) -> float:
    return metric(predictions, targets)

def mean_error(predictions: list[float], targets: list[float]) -> float:
    errors = [pred - target for pred, target in zip(predictions, targets)]
    return sum(errors) / len(errors)

assert apply_metric(mean_error, [2.0, 4.0], [1.0, 1.0]) == 2.0
print("checks passed")

The annotations say what shapes of Python objects the function expects. `Callable[[...], float]` means the argument should be a function that accepts the listed argument types and returns a float.

#### 3. Modify Drills

**Modify Drill 1.** Change the return type from `int` to `str` after changing the function body.

In [ ]:
def make_id(number: int) -> str:
    return "id-" + str(number)

assert make_id.__annotations__ == {"number": int, "return": str}
assert make_id(7) == "id-7"
print("passed")

**Modify Drill 2.** Use `str | None` for an optional nickname.

In [ ]:
def display_name(name: str, nickname: str | None) -> str:
    if nickname is None:
        return name
    return name + " (" + nickname + ")"

assert display_name("Ada", None) == "Ada"
assert display_name("Ada", "A") == "Ada (A)"
print("passed")

**Modify Drill 3.** Type a nested container and verify the annotation is present.

In [ ]:
def first_row(matrix: list[list[float]]) -> list[float]:
    return matrix[0]

assert first_row([[1.0, 2.0]]) == [1.0, 2.0]
assert "matrix" in first_row.__annotations__
print("passed")

#### 4. Break-and-Fix Drills

**Break-and-Fix Drill 1.** Break it by annotating the return as `int` while returning a string. Predict why Python still runs, then fix the misleading hint.

In [ ]:
def label(number: int) -> str:
    return "label-" + str(number)

actual = label(3)
assert actual == "label-3"
assert label.__annotations__["return"] is str
print("passed")

**Break-and-Fix Drill 2.** Break it by using `list` without the contained type. Predict what information reviewers and tools lose, then restore `list[int]`.

In [ ]:
def total(values: list[int]) -> int:
    return sum(values)

assert total([1, 2, 3]) == 6
assert total.__annotations__["values"] == list[int]
print("passed")

#### 5. Self-Verification
Use runtime behavior asserts plus annotation asserts. Type hints are mechanically present in `function.__annotations__`, even though Python does not enforce them for ordinary calls.

#### 6. Standalone Exercises

**Exercise 1.** Add parameter and return annotations. Expected behavior: annotation dictionary matches.

In [ ]:
def add(a, b):
    return a + b

assert add(2, 3) == 5
assert add.__annotations__ == {"a": int, "b": int, "return": int}
print("passed")

**Exercise 2.** Annotate a function that may return `None`. Expected behavior: return annotation is `float | None`.

In [ ]:
def safe_first(values):
    if len(values) == 0:
        return None
    return values[0]

assert safe_first([2.5]) == 2.5
assert safe_first([]) is None
assert safe_first.__annotations__["return"] == float | None
print("passed")

**Exercise 3.** Annotate a callable argument. Expected behavior: result is `[2, 4]`.

In [ ]:
from typing import Callable

def apply_each(func, values):
    return [func(value) for value in values]

def double(x: int) -> int:
    return x * 2

assert apply_each(double, [1, 2]) == [2, 4]
assert "func" in apply_each.__annotations__
print("passed")

**Exercise 4.** Annotate a dictionary from strings to floats. Expected behavior: total is `0.3`.

In [ ]:
def sum_metrics(metrics):
    return sum(metrics.values())

actual = sum_metrics({"loss": 0.1, "mae": 0.2})
assert round(actual, 2) == 0.3
assert sum_metrics.__annotations__["metrics"] == dict[str, float]
print("passed")

**Exercise 5.** Annotate a tuple return. Expected behavior: `('loss', 0.5)`.

In [ ]:
def metric_pair(name, value):
    return name, value

actual = metric_pair("loss", 0.5)
assert actual == ("loss", 0.5)
assert metric_pair.__annotations__["return"] == tuple[str, float]
print("passed")

#### 7. Applied AI/ML Drill
**ML to Python mirror:** a typed training step is just a function contract for a metric function, numeric inputs, and a numeric result. **Python to ML mirror:** ML codebases use type hints on data loaders, metric callbacks, and training utilities so readers can see whether a function expects batches, labels, callables, or scalar metrics.

**Applied Drill.** Add precise annotations to `train_step`. Expected behavior: loss `0.5` and a typed `metric` argument.

In [ ]:
from typing import Callable

def absolute_error(prediction: float, target: float) -> float:
    return abs(prediction - target)

def train_step(weight, x, target, metric):
    prediction = weight * x
    return metric(prediction, target)

actual = train_step(2.0, 3.0, 5.5, absolute_error)
assert actual == 0.5
assert "metric" in train_step.__annotations__
print("passed")

#### 8. Common Bugs
- Expecting Python to enforce hints automatically: the symptom is a wrong type still running until the code itself fails.
- Writing vague containers such as `list` or `dict`: the symptom is a signature that hides what the contents should be.
- Forgetting `None` in optional returns: the symptom is a caller that assumes a value is always present.
- Over-annotating simple local variables: the symptom is clutter that makes the important function contract harder to see.

#### 9. Compounding Drill

Combine type hints with decorators: annotate a decorator that logs a float-returning function while preserving behavior.

In [ ]:
from typing import Callable
from functools import wraps

def log_float(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print("value:", result)
        return result
    return wrapper

@log_float
def score(x: float) -> float:
    return x * 2

assert score(1.5) == 3.0
assert "func" in log_float.__annotations__
print("passed")

#### 10. Chapter Project
No chapter project in this chapter. Projects appear every 2-3 chapters so the longer drills can compound several mechanics at once.

#### 11. Solutions Appendix

--- SOLUTIONS: DO NOT PEEK UNTIL ATTEMPTED ---

In [ ]:
# Standalone Exercises 1-3
from typing import Callable

def add(a: int, b: int) -> int:
    return a + b

assert add.__annotations__ == {"a": int, "b": int, "return": int}

def safe_first(values: list[float]) -> float | None:
    if len(values) == 0:
        return None
    return values[0]

assert safe_first.__annotations__["return"] == float | None

def apply_each(func: Callable[[int], int], values: list[int]) -> list[int]:
    return [func(value) for value in values]

def double(x: int) -> int:
    return x * 2

assert apply_each(double, [1, 2]) == [2, 4]
print("solutions passed")

In [ ]:
# Standalone Exercises 4-5, Applied Drill, and Compounding Drill
from typing import Callable
from functools import wraps

def sum_metrics(metrics: dict[str, float]) -> float:
    return sum(metrics.values())

assert round(sum_metrics({"loss": 0.1, "mae": 0.2}), 2) == 0.3

def metric_pair(name: str, value: float) -> tuple[str, float]:
    return name, value

assert metric_pair("loss", 0.5) == ("loss", 0.5)

def absolute_error(prediction: float, target: float) -> float:
    return abs(prediction - target)

def train_step(weight: float, x: float, target: float,
               metric: Callable[[float, float], float]) -> float:
    prediction = weight * x
    return metric(prediction, target)

assert train_step(2.0, 3.0, 5.5, absolute_error) == 0.5

def log_float(func: Callable[..., float]) -> Callable[..., float]:
    @wraps(func)
    def wrapper(*args, **kwargs) -> float:
        result = func(*args, **kwargs)
        print("value:", result)
        return result
    return wrapper

print("solutions passed")